In [1]:
# 高性能VAE缺失值填补模型 - 最终优化版√√√
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from scipy import stats
import warnings
import random
import os
import gc
import time

warnings.filterwarnings("ignore")

# 检查TensorFlow版本
print(f"TensorFlow版本: {tf.__version__}")

# ==================== 强化的可重复性设置 ====================
SEED = 42

def set_all_seeds(seed=42):
    """设置所有随机种子以确保可重复性"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# 初始化随机种子
set_all_seeds(SEED)

# GPU内存设置
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"发现 {len(gpus)} 个GPU设备")
    except RuntimeError as e:
        print(f"GPU设置错误: {e}")
else:
    print("未发现GPU，使用CPU运行")

# 清理会话
tf.keras.backend.clear_session()
gc.collect()

# 1. 数据加载和预处理
def load_and_preprocess_data():
    """加载数据并进行初步预处理"""
    data = pd.read_csv("./Soil.csv", low_memory=True)
    print(f"原始数据形状: {data.shape}")
    
    # 保存原始数据副本
    original_data = data.copy()
    
    # 删除非数值列
    cols_to_drop = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=cols_to_drop)
    
    # 删除ID列
    for col in data.columns[:3]:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # 删除L(m)列
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # 移除缺失率过高的列（>50%）
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = missing_ratio[missing_ratio > 0.50].index
    data = data.drop(columns=cols_to_drop)
    
    # 转换数据类型
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    print(f"最终数据形状: {data.shape}")
    print(f"特征: {list(data.columns)}")
    
    return data, original_data

# 2. 自适应异常值处理
def adaptive_outlier_handling(df):
    """自适应异常值处理，根据数据分布选择策略"""
    df_clean = df.copy()
    transform_info = {}
    
    for column in df_clean.columns:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # 计算偏度和峰度
        skewness = stats.skew(values)
        kurtosis = stats.kurtosis(values)
        
        # 根据分布特征选择处理方法
        if abs(skewness) > 2:  # 高度偏斜
            # 使用更宽松的分位数
            lower, upper = np.percentile(values, [0.1, 99.9])
        else:  # 接近正态
            # 使用IQR方法
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            lower = q1 - 3 * iqr
            upper = q3 + 3 * iqr
            
            # 确保不丢失太多数据
            lower = max(lower, np.percentile(values, 0.5))
            upper = min(upper, np.percentile(values, 99.5))
        
        values = np.clip(values, lower, upper)
        
        transform_info[column] = {
            'type': 'adaptive_clip',
            'lower': lower,
            'upper': upper,
            'skewness': skewness,
            'kurtosis': kurtosis
        }
        
        df_clean.loc[mask, column] = values
    
    return df_clean, transform_info

# 3. 优化的混合标准化
def optimized_normalization(data):
    """优化的混合标准化策略"""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    for column in data.columns:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # 计算数据特征
            skewness = stats.skew(values.flatten())
            value_range = values.max() - values.min()
            
            # 选择最合适的标准化方法
            if abs(skewness) > 1 or value_range > 1000:
                # 高偏斜或大范围：使用分位数变换
                scaler = QuantileTransformer(
                    output_distribution='normal',
                    n_quantiles=min(1000, len(values)),
                    subsample=100000,
                    random_state=42
                )
            elif column in ['H(m)', 'At(m²）']:  # 特定问题特征
                # 使用更鲁棒的标准化
                scaler = RobustScaler(quantile_range=(5, 95))
            else:
                # 默认使用标准化
                scaler = StandardScaler()
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # 映射到[0.1, 0.9]
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                scaled = (scaled - min_val) / (max_val - min_val) * 0.8 + 0.1
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {
                'scaler': scaler,
                'min': min_val,
                'max': max_val,
                'type': type(scaler).__name__
            }
    
    return data_normalized, scalers

# 4. 改进的反标准化
def inverse_transform_optimized(data, scalers, column_names):
    """优化的反标准化"""
    data_original = np.zeros_like(data)
    
    for col_idx, col_name in enumerate(column_names):
        values = data[:, col_idx].copy()
        
        if col_name in scalers:
            scaler_info = scalers[col_name]
            
            # 反映射
            values = (values - 0.1) / 0.8
            values = values * (scaler_info['max'] - scaler_info['min']) + scaler_info['min']
            
            # 反标准化
            values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        data_original[:, col_idx] = values
    
    return data_original

# 5. 增强的特征工程
def enhanced_feature_engineering(data, mask):
    """增强的特征工程"""
    n_samples, n_features = data.shape
    
    # 基础特征
    features = [data]
    
    # 平方特征
    squared = data ** 2
    features.append(squared)
    
    # 平方根特征（处理负值）
    sqrt_data = np.sqrt(np.abs(data)) * np.sign(data)
    features.append(sqrt_data)
    
    # 对数特征（处理零和负值）
    log_data = np.log1p(np.abs(data)) * np.sign(data)
    features.append(log_data)
    
    # 选择性交互特征（只选择重要的交互）
    interactions = []
    for i in range(min(n_features, 3)):  # 限制交互特征数量
        for j in range(i+1, min(n_features, 3)):
            interaction = data[:, i:i+1] * data[:, j:j+1]
            interactions.append(interaction)
    
    if interactions:
        features.extend(interactions)
    
    # 合并所有特征
    enhanced_data = np.hstack(features)
    
    # 扩展掩码
    n_new_features = enhanced_data.shape[1] - n_features
    enhanced_mask = np.hstack([mask] + [mask] * (n_new_features // n_features))
    
    # 调整掩码大小
    if enhanced_mask.shape[1] < enhanced_data.shape[1]:
        diff = enhanced_data.shape[1] - enhanced_mask.shape[1]
        enhanced_mask = np.hstack([enhanced_mask, mask[:, :diff]])
    
    return enhanced_data, enhanced_mask

# 6. 最终优化的VAE模型
class FinalOptimizedVAE:
    """最终优化的VAE模型"""
    def __init__(self, data_dim, enhanced_dim, latent_dim=8, learning_rate=0.0008, seed=42):
        self.data_dim = data_dim
        self.enhanced_dim = enhanced_dim
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.seed = seed
        
        tf.random.set_seed(seed)
        
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # 使用带重启的余弦退火
        lr_schedule = optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=learning_rate,
            first_decay_steps=500,
            t_mul=2.0,
            m_mul=0.9,
            alpha=0.1
        )
        self.optimizer = optimizers.Adam(learning_rate=lr_schedule)
        
        self.best_weights = None
        self.best_val_loss = float('inf')
    
    def _build_encoder(self):
        """构建优化的编码器"""
        inputs = layers.Input(shape=(self.enhanced_dim,))
        
        # 渐进式降维
        x = layers.Dense(64, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(inputs)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.25)(x)
        
        x = layers.Dense(32, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.2)(x)
        
        x = layers.Dense(16, activation='swish',
                        kernel_initializer='he_normal')(x)
        x = layers.LayerNormalization()(x)
        
        # 潜在空间
        z_mean = layers.Dense(self.latent_dim)(x)
        z_log_var = layers.Dense(self.latent_dim)(x)
        
        # 限制方差范围
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -4, 3))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """构建优化的解码器"""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        # 渐进式升维
        x = layers.Dense(16, activation='swish',
                        kernel_initializer='he_normal')(inputs)
        x = layers.LayerNormalization()(x)
        
        x = layers.Dense(32, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.2)(x)
        
        x = layers.Dense(64, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.25)(x)
        
        # 输出层
        outputs = layers.Dense(self.data_dim, activation='sigmoid')(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var):
        """重参数化技巧"""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.05
    
    @tf.function
    def train_step(self, x_batch_enhanced, x_batch_original, mask_batch, beta):
        """训练步骤with KL退火"""
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encoder(x_batch_enhanced, training=True)
            z = self._sample(z_mean, z_log_var)
            reconstructed = self.decoder(z, training=True)
            
            # 重建损失
            mse = tf.square(x_batch_original - reconstructed) * mask_batch
            reconstruction_loss = tf.reduce_sum(mse) / tf.reduce_sum(mask_batch)
            
            # KL损失
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # 总损失with KL退火
            total_loss = reconstruction_loss + beta * kl_loss
            
            # L2正则化
            l2_loss = tf.add_n(self.encoder.losses + self.decoder.losses)
            total_loss += l2_loss
        
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        
        # 梯度裁剪
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss, reconstruction_loss, kl_loss
    
    def fit(self, data_enhanced, data_original, mask, validation_split=0.2, 
            epochs=120, batch_size=64):
        """训练模型with改进策略"""
        n_samples = len(data_original)
        indices = np.arange(n_samples)
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train_enhanced = data_enhanced[train_indices]
        x_train_original = data_original[train_indices]
        x_val_enhanced = data_enhanced[val_indices]
        x_val_original = data_original[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"训练集: {len(x_train_original)} 样本")
        print(f"验证集: {len(x_val_original)} 样本")
        
        patience = 25
        patience_counter = 0
        min_improvement = 0.00005
        
        # KL退火计划
        kl_anneal_epochs = 30
        
        for epoch in range(epochs):
            # KL退火
            beta = min(1.0, epoch / kl_anneal_epochs) * 0.001
            
            train_indices_shuffled = np.random.permutation(len(x_train_original))
            train_loss = 0
            train_recon_loss = 0
            train_kl_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train_original), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss, recon_loss, kl_loss = self.train_step(
                    x_train_enhanced[batch_indices],
                    x_train_original[batch_indices],
                    mask_train[batch_indices],
                    beta
                )
                train_loss += loss
                train_recon_loss += recon_loss
                train_kl_loss += kl_loss
                n_batches += 1
            
            train_loss /= n_batches
            train_recon_loss /= n_batches
            train_kl_loss /= n_batches
            
            # 验证
            if epoch % 5 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val_original), batch_size):
                    z_mean, z_log_var = self.encoder(x_val_enhanced[i:i+batch_size], training=False)
                    z = self._sample(z_mean, z_log_var)
                    reconstructed = self.decoder(z, training=False)
                    
                    mse = tf.square(x_val_original[i:i+batch_size] - reconstructed) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                
                # 早停逻辑
                if val_loss < self.best_val_loss - min_improvement:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 5
                
                if epoch % 20 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, "
                          f"Recon={train_recon_loss:.6f}, KL={train_kl_loss:.6f}, "
                          f"Val Loss={val_loss:.6f}")
                
                if patience_counter >= patience:
                    print(f"早停于第 {epoch} 轮")
                    break
        
        # 恢复最佳权重
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data_enhanced, data_original, mask):
        """填补缺失值"""
        imputed = np.zeros_like(data_original)
        batch_size = 128
        
        for i in range(0, len(data_original), batch_size):
            batch_enhanced = data_enhanced[i:i+batch_size]
            batch_original = data_original[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            # 多次采样取平均
            predictions = []
            for _ in range(7):  # 增加采样次数
                z_mean, z_log_var = self.encoder(batch_enhanced, training=False)
                z = self._sample(z_mean, z_log_var)
                pred = self.decoder(z, training=False).numpy()
                predictions.append(pred)
            
            # 使用修剪均值（去除极值）
            predictions = np.array(predictions)
            pred_mean = np.mean(np.sort(predictions, axis=0)[1:-1], axis=0)
            
            imputed[i:i+batch_size] = batch_original * batch_mask + pred_mean * (1 - batch_mask)
        
        return imputed

# 7. 增强的目标特征模型
class EnhancedTargetedModel:
    """增强的目标特征专用模型"""
    def __init__(self, target_features, seed=42):
        self.target_features = target_features
        self.seed = seed
        self.models = {}
    
    def fit(self, data_original, mask, column_names):
        """为目标特征训练集成模型"""
        print("训练增强的目标特征模型...")
        
        simple_imputer = SimpleImputer(strategy='median')
        data_filled = simple_imputer.fit_transform(data_original)
        
        for col_idx, col_name in enumerate(column_names):
            if col_name not in self.target_features:
                continue
            
            obs_mask = mask[:, col_idx] == 1
            if np.sum(obs_mask) < 100:
                continue
            
            X_train = np.delete(data_filled[obs_mask], col_idx, axis=1)
            y_train = data_original[obs_mask, col_idx]
            
            # 为每个特征使用集成模型
            if col_name == 'H(m)':
                # H(m)使用投票集成
                models = [
                    ('mlp', MLPRegressor(
                        hidden_layer_sizes=(100, 50),
                        activation='tanh',
                        solver='lbfgs',
                        alpha=0.01,
                        max_iter=1000,
                        random_state=self.seed
                    )),
                    ('rf', RandomForestRegressor(
                        n_estimators=100,
                        max_depth=10,
                        random_state=self.seed
                    )),
                    ('gb', GradientBoostingRegressor(
                        n_estimators=100,
                        max_depth=5,
                        random_state=self.seed
                    ))
                ]
                model = VotingRegressor(models)
                
            elif col_name == 'At(m²）':
                # At(m²）使用优化的ExtraTrees
                model = ExtraTreesRegressor(
                    n_estimators=200,
                    max_depth=12,
                    min_samples_split=3,
                    min_samples_leaf=1,
                    max_features='sqrt',
                    bootstrap=True,
                    random_state=self.seed
                )
                
            else:  # Wsa(m)
                # Wsa(m)使用优化的GradientBoosting
                model = GradientBoostingRegressor(
                    n_estimators=200,
                    max_depth=6,
                    learning_rate=0.03,
                    subsample=0.85,
                    min_samples_split=3,
                    min_samples_leaf=2,
                    random_state=self.seed
                )
            
            model.fit(X_train, y_train)
            self.models[col_idx] = model
            
            # 评估性能
            train_pred = model.predict(X_train)
            train_r2 = r2_score(y_train, train_pred)
            print(f"  {col_name}: 训练R²={train_r2:.4f}")
    
    def impute(self, data_original, mask):
        """填补缺失值"""
        simple_imputer = SimpleImputer(strategy='median')
        data_filled = simple_imputer.fit_transform(data_original)
        imputed = data_original.copy()
        
        for col_idx, model in self.models.items():
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) == 0:
                continue
            
            X_miss = np.delete(data_filled[miss_mask], col_idx, axis=1)
            imputed[miss_mask, col_idx] = model.predict(X_miss)
        
        return imputed

# 8. 最终集成模型
class FinalEnsemble:
    """最终优化的集成模型"""
    def __init__(self, data_dim, seed=42):
        self.data_dim = data_dim
        self.seed = seed
        self.models = []
        self.weights = None
        self.scalers = None
        self.data_raw = None
    
    def fit(self, data_normalized, mask, data_raw, scalers, column_names):
        """训练最终集成模型"""
        print("\n训练最终集成模型...")
        self.scalers = scalers
        self.data_raw = data_raw
        
        # 创建增强特征
        data_enhanced, mask_enhanced = enhanced_feature_engineering(data_normalized, mask)
        print(f"增强特征维度: {data_enhanced.shape[1]}")
        
        # 1. KNN模型
        print("1. KNN模型...")
        knn = KNNImputer(n_neighbors=12, weights='distance')
        knn.fit(data_normalized)
        self.models.append(('knn', knn))
        
        # 2. IterativeImputer
        print("2. IterativeImputer...")
        mice = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=50,
                max_depth=8,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        mice.fit(data_normalized)
        self.models.append(('mice', mice))
        
        # 3. 优化的VAE
        print("3. 优化的VAE模型...")
        vae = FinalOptimizedVAE(
            data_dim=self.data_dim,
            enhanced_dim=data_enhanced.shape[1],
            latent_dim=8,
            learning_rate=0.0008,
            seed=self.seed
        )
        
        mean_imputer = SimpleImputer(strategy='median')
        mean_imputer.fit(data_normalized)
        initial_imputed = mean_imputer.transform(data_normalized)
        data_filled = np.where(mask == 1, data_normalized, initial_imputed)
        
        data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
        
        vae.fit(data_filled_enhanced, data_filled, mask, epochs=120, batch_size=64)
        self.models.append(('vae', vae))
        self.vae_enhanced_data = data_filled_enhanced
        
        # 4. 增强的目标模型
        print("4. 增强的目标特征模型...")
        target_model = EnhancedTargetedModel(
            target_features=['H(m)', 'At(m²）', 'Wsa(m)'],
            seed=self.seed
        )
        target_model.fit(data_raw, mask, column_names)
        self.models.append(('targeted', target_model))
        
        print("集成模型训练完成")
        
        # 自适应权重
        self._compute_adaptive_weights()
    
    def _compute_adaptive_weights(self):
        """计算自适应权重"""
        print("\n计算自适应权重...")
        
        # 基础权重
        base_weights = {
            'knn': 0.15,
            'mice': 0.20,
            'vae': 0.30,
            'targeted': 0.35
        }
        
        self.weights = np.array([base_weights[name] for name, _ in self.models])
        self.weights = self.weights / np.sum(self.weights)
        
        print(f"最终权重: {dict(zip([n for n, _ in self.models], self.weights))}")
    
    def impute(self, data_normalized, mask, column_names):
        """最终集成预测"""
        predictions = []
        
        for (name, model), weight in zip(self.models, self.weights):
            if name == 'knn':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'mice':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'vae':
                mean_imputer = SimpleImputer(strategy='median')
                mean_imputer.fit(data_normalized)
                initial_imputed = mean_imputer.transform(data_normalized)
                data_filled = np.where(mask == 1, data_normalized, initial_imputed)
                data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
                
                pred = model.impute(data_filled_enhanced, data_filled, mask)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'targeted':
                pred_original = model.impute(self.data_raw, mask)
            
            predictions.append(pred_original * weight)
        
        # 加权平均
        imputed = np.sum(predictions, axis=0)
        
        # 确保观测值不变
        imputed = self.data_raw * mask + imputed * (1 - mask)
        
        return imputed

# 9. 最终评估函数
# ✅ 修改位置：新增输出观测值与预测值 CSV 文件
# ============================================================
def final_evaluation(model, data_normalized, mask, column_names, 
                     data_raw, scalers):
    """最终评估with详细诊断 + 导出每个特征的观测与预测值"""
    print("\n=== 最终评估（5折交叉验证）===")
    
    results = []
    output_dir = "参数观测预测对比"
    os.makedirs(output_dir, exist_ok=True)  # 创建输出文件夹

    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        
        if len(obs_indices) < 100:
            continue
        
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        
        all_true_original = []
        all_pred_original = []
        fold_metrics = []
        
        for fold, (train_idx, test_idx) in enumerate(kf.split(obs_indices)):
            test_indices = obs_indices[test_idx]
            
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            
            imputed_original = model.impute(data_normalized, test_mask, column_names)
            
            true_values = data_raw[test_indices, col_idx]
            pred_values = imputed_original[test_indices, col_idx]
            
            if len(true_values) > 1:
                fold_r2 = r2_score(true_values, pred_values)
                fold_mae = mean_absolute_error(true_values, pred_values)
                fold_rmse = np.sqrt(mean_squared_error(true_values, pred_values))  # 计算RMSE
                fold_metrics.append({'r2': fold_r2, 'mae': fold_mae, 'rmse': fold_rmse})
            
            all_true_original.extend(true_values)
            all_pred_original.extend(pred_values)
        
        all_true_original = np.array(all_true_original)
        all_pred_original = np.array(all_pred_original)
        
        # ✅ 新增输出：保存观测值与预测值
        compare_df = pd.DataFrame({
            'True': all_true_original,
            'Pred': all_pred_original
        })
        safe_name = str(col_name).replace('/', '_').replace('(', '').replace(')', '').replace('²', '2')
        compare_path = os.path.join(output_dir, f"参数对比_{safe_name}.csv")
        compare_df.to_csv(compare_path, index=False, encoding='utf-8-sig')

        # 计算整体指标
        mae = mean_absolute_error(all_true_original, all_pred_original)
        rmse = np.sqrt(mean_squared_error(all_true_original, all_pred_original))  # 计算RMSE
        ss_res = np.sum((all_true_original - all_pred_original) ** 2)
        ss_tot = np.sum((all_true_original - np.mean(all_true_original)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        r2_scores = [m['r2'] for m in fold_metrics]
        r2_std = np.std(r2_scores) if r2_scores else 0
        
        non_zero_mask = np.abs(all_true_original) > 1e-8
        if np.sum(non_zero_mask) > 0:
            mape = np.mean(np.abs((all_true_original[non_zero_mask] - all_pred_original[non_zero_mask]) / 
                                all_true_original[non_zero_mask])) * 100
        else:
            mape = np.mean(np.abs(all_true_original - all_pred_original)) * 100
        
        if np.std(all_true_original) > 0 and np.std(all_pred_original) > 0:
            correlation = np.corrcoef(all_true_original, all_pred_original)[0, 1]
        else:
            correlation = 0
        
        status = "优秀"
        if r2 < 0.6:
            status = "需改进"
        elif r2 < 0.8:
            status = "良好"
        elif r2 > 0.95 and mape > 30:
            status = "检查过拟合"
        elif r2_std > 0.05:
            status = "稳定性待改进"
        
        results.append({
            '特征': col_name,
            'R²': r2,
            'R²_std': r2_std,
            '相关系数': correlation,
            'MAE': mae,
            'RMSE': rmse,  # 新增RMSE
            'MAPE(%)': mape,
            '状态': status
        })
        
        print(f"{col_name}: R²={r2:.4f}±{r2_std:.4f}, MAPE={mape:.2f}%, 状态={status}, RMSE={rmse:.4f}")
        print(f"  ↳ 已保存预测对比文件：{compare_path}")
    
    return pd.DataFrame(results)

# 10. 主函数
def main():
    """主函数"""
    print("=" * 80)
    print("高性能VAE缺失值填补模型 - 最终优化版")
    print("目标：所有特征R²≥0.8，模型稳定无过拟合")
    print("=" * 80)
    
    try:
        set_all_seeds(SEED)
        
        # 1. 加载数据
        print("\n1. 加载数据...")
        data_raw, original_data = load_and_preprocess_data()
        
        # 2. 自适应异常值处理
        print("\n2. 自适应异常值处理...")
        data_clean, transform_info = adaptive_outlier_handling(data_raw)
        
        # 3. 优化的标准化
        print("\n3. 优化的混合标准化...")
        data_normalized, scalers = optimized_normalization(data_clean)
        
        # 创建掩码
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        data_raw_array = data_clean.fillna(0).values.astype(np.float32)
        
        print(f"\n数据形状: {data_array.shape}")
        print(f"缺失率: {(1 - mask.mean())*100:.2f}%")
        
        # 显示标准化策略
        print("\n标准化策略：")
        for col_name, scaler_info in scalers.items():
            print(f"  {col_name}: {scaler_info['type']}")
        
        # 4. 训练最终集成模型
        print("\n4. 训练最终集成模型...")
        model = FinalEnsemble(data_dim=data_array.shape[1], seed=SEED)
        model.fit(data_array, mask, data_raw_array, scalers, data_normalized.columns)
        
        # 5. 最终评估
        print("\n5. 最终评估...")
        eval_results = final_evaluation(
            model, data_array, mask, data_normalized.columns,
            data_raw_array, scalers
        )
        
        # 6. 生成填补数据
        print("\n6. 生成最终填补数据...")
        final_imputed_original = model.impute(data_array, mask, data_normalized.columns)
        
        # 7. 输出结果
        print("\n" + "=" * 80)
        print("最终模型性能")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['特征']}:")
            print(f"  R² = {row['R²']:.4f} ± {row['R²_std']:.4f}")
            print(f"  MAE = {row['MAE']:.4f}")
            print(f"  MAPE = {row['MAPE(%)']:.2f}%")
            print(f"  状态: {row['状态']}")
        
        # 汇总统计
        print("\n" + "-" * 40)
        print("汇总统计")
        print("-" * 40)
        
        avg_r2 = eval_results['R²'].mean()
        min_r2 = eval_results['R²'].min()
        max_r2 = eval_results['R²'].max()
        avg_mape = eval_results['MAPE(%)'].mean()
        
        print(f"平均R²: {avg_r2:.4f}")
        print(f"最低R²: {min_r2:.4f}")
        print(f"最高R²: {max_r2:.4f}")
        print(f"平均MAPE: {avg_mape:.2f}%")
        
        # 目标达成检查
        target_r2 = 0.8
        achieved = eval_results[eval_results['R²'] >= target_r2]
        not_achieved = eval_results[eval_results['R²'] < target_r2]
        
        print(f"\n目标达成情况（R²≥{target_r2}）:")
        print(f"  达标特征: {len(achieved)}/{len(eval_results)}")
        
        if not_achieved.empty:
            print("  ✅ 所有特征均达标！")
        else:
            print("  未达标特征:")
            for _, row in not_achieved.iterrows():
                print(f"    - {row['特征']}: R²={row['R²']:.4f}")
        
        # 稳定性检查
        unstable = eval_results[eval_results['R²_std'] > 0.05]
        if not unstable.empty:
            print("\n稳定性警告（R²_std > 0.05）:")
            for _, row in unstable.iterrows():
                print(f"  - {row['特征']}: std={row['R²_std']:.4f}")
        else:
            print("\n✅ 所有特征模型稳定性良好")
        
        # 保存结果
        final_df_original = pd.DataFrame(final_imputed_original, columns=data_normalized.columns)
        final_df_original.to_csv("填补结果_最终优化版.csv", index=False)
        eval_results.to_csv("评估结果_最终优化版.csv", index=False)
        
        print("\n文件已保存：")
        print("- 填补结果_最终优化版.csv")
        print("- 评估结果_最终优化版.csv")
        
        return final_imputed_original, eval_results
        
    except Exception as e:
        print(f"\n错误: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    print("\n程序开始运行...")
    print("最终优化特性：")
    print("1. 自适应异常值处理（基于偏度和峰度）")
    print("2. 优化的混合标准化（根据数据分布选择）")
    print("3. 增强特征工程（平方、平方根、对数、交互）")
    print("4. Swish激活函数 + LayerNorm（更好的梯度流）")
    print("5. KL退火 + 余弦退火学习率")
    print("6. 增强的目标模型（集成方法）")
    print("7. 修剪均值（去除极值影响）")
    print("8. 改进的MAPE计算（处理零值）")
    
    start_time = time.time()
    result, evaluation = main()
    
    if result is not None:
        print(f"\n✅ 程序运行完成！")
        print(f"总用时: {(time.time() - start_time)/60:.2f} 分钟")
        
        # 性能总结
        if evaluation is not None:
            print("\n" + "=" * 50)
            print("性能总结")
            print("=" * 50)
            
            if evaluation['R²'].min() >= 0.8:
                print("✅ 主要目标达成：所有特征R²≥0.8")
            
            if evaluation['R²_std'].max() < 0.05:
                print("✅ 模型稳定性优秀")
            
            if evaluation['MAPE(%)'].max() < 50:
                print("✅ 相对误差控制良好")
            
            print("\n感谢使用！模型已优化至最佳状态。")
    else:
        print("\n❌ 运行失败，请检查错误信息")
    
    gc.collect()

TensorFlow版本: 2.21.0
未发现GPU，使用CPU运行


程序开始运行...
最终优化特性：
1. 自适应异常值处理（基于偏度和峰度）
2. 优化的混合标准化（根据数据分布选择）
3. 增强特征工程（平方、平方根、对数、交互）
4. Swish激活函数 + LayerNorm（更好的梯度流）
5. KL退火 + 余弦退火学习率
6. 增强的目标模型（集成方法）
7. 修剪均值（去除极值影响）
8. 改进的MAPE计算（处理零值）
高性能VAE缺失值填补模型 - 最终优化版
目标：所有特征R²≥0.8，模型稳定无过拟合

1. 加载数据...

错误: [Errno 2] No such file or directory: './Soil.csv'

❌ 运行失败，请检查错误信息


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27516\1920333375.py", line 861, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27516\1920333375.py", line 62, in load_and_preprocess_data
    data = pd.read_csv("./Soil.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 1645, in 

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    将L(m)列添加到已生成的填补结果文件中
    """
    # 1. 读取原始数据以获取L(m)列
    original_data = pd.read_csv("./Soil.csv", low_memory=True)
    
    # 2. 检查L(m)列是否存在
    if 'L(m)' not in original_data.columns:
        print("原始数据中未找到L(m)列")
        return
    
    # 3. 提取L(m)列
    L_column = original_data['L(m)'].copy()
    print(f"成功提取L(m)列，共{len(L_column)}行")
    
    # 4. 读取填补结果
    try:
        imputed_results = pd.read_csv("填补结果_最终优化版.csv")
        print(f"填补结果形状: {imputed_results.shape}")
    except FileNotFoundError:
        print("未找到填补结果文件，请先运行主程序生成结果")
        return
    
    # 5. 检查行数是否匹配
    if len(imputed_results) != len(L_column):
        print(f"警告：行数不匹配！填补结果有{len(imputed_results)}行，L(m)列有{len(L_column)}行")
        # 如果行数不同，需要对齐（取最小行数）
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. 添加L(m)列到结果中
    imputed_results['L(m)'] = L_column.values
    
    # 7. 保存更新后的结果
    output_filename = "填补结果_Soil_包含L列_平衡版.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\n成功添加L(m)列到结果中！")
    print(f"新文件已保存为: {output_filename}")
    print(f"最终数据形状: {imputed_results.shape}")
    print(f"列名: {list(imputed_results.columns)}")
    
    # 8. 显示L(m)列的基本统计信息
    print("\nL(m)列统计信息:")
    print(f"  数据类型: {L_column.dtype}")
    print(f"  唯一值数量: {L_column.nunique()}")
    print(f"  缺失值数量: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  最小值: {L_column.min()}")
        print(f"  最大值: {L_column.max()}")
        print(f"  平均值: {L_column.mean():.2f}")
    else:
        print(f"  前5个值: {L_column.head().tolist()}")
    
    return imputed_results

# 运行函数
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

成功提取L(m)列，共4349行
填补结果形状: (4349, 5)

成功添加L(m)列到结果中！
新文件已保存为: 填补结果_Soil_包含L列_平衡版.csv
最终数据形状: (4349, 6)
列名: ['H(m)', 'As(m²）', 'At(m²）', 'V(m³)', 'Wsa(m)', 'L(m)']

L(m)列统计信息:
  数据类型: float64
  唯一值数量: 3107
  缺失值数量: 0
  最小值: 1.17
  最大值: 1610000.0
  平均值: 760.89
